# 01 Sensor Data Preparation

## Purpose
This notebook prepares the City of Melbourne parking sensor datasets (2011–2020) for the project. It combines the yearly datasets into a single dataset, standardises the data structure across all years, filters the records to the project streets, and performs basic data quality checks. The output of this notebook is used in the next notebook for data cleaning and exploratory data analysis (EDA).

### Output
- `parking_sensor_2011_2020.csv`
- `parking_sensor_project_streets.csv`

The output of this notebook is used in `02_sensor_cleaning_eda_duckdb.ipynb`.

In [2]:
from pathlib import Path
import pandas as pd
import re

### 1. Load 2011–2020 Sensor Datasets
Load the yearly parking sensor datasets (2011–2020) into the notebook for data preparation.

In [3]:
# Find the project folder
project_path = Path.cwd().resolve().parents[2]

# Set the raw data folder
data_folder = project_path / "data" / "raw"

# Find only the parking sensor CSV files
sensor_files = sorted(
    data_folder.glob("On-street_Car_Parking_Sensor_Data*.csv")
)

# Display the project information
print(f"Project folder: {project_path}")
print(f"Raw data folder: {data_folder}")
print(f"Number of sensor files: {len(sensor_files)}")

for file_path in sensor_files:
    print(file_path.name)

Project folder: /Users/thanya/Desktop/Victoria-Urban-Planning
Raw data folder: /Users/thanya/Desktop/Victoria-Urban-Planning/data/raw
Number of sensor files: 10
On-street_Car_Parking_Sensor_Data_-_2011.csv
On-street_Car_Parking_Sensor_Data_-_2012.csv
On-street_Car_Parking_Sensor_Data_-_2013.csv
On-street_Car_Parking_Sensor_Data_-_2014.csv
On-street_Car_Parking_Sensor_Data_-_2015.csv
On-street_Car_Parking_Sensor_Data_-_2016.csv
On-street_Car_Parking_Sensor_Data_-_2017.csv
On-street_Car_Parking_Sensor_Data_-_2018.csv
On-street_Car_Parking_Sensor_Data_-_2019.csv
On-street_Car_Parking_Sensor_Data_-_2020__Jan_-_May_.csv


### 2. Schema Validation (Column Match)
Check the column names in each yearly parking sensor dataset (2011–2020) to identify any differences before combining the datasets.

In [4]:
# Stop if no sensor files are available
if not sensor_files:
    raise FileNotFoundError(
        "No parking sensor CSV files were found. "
        "Run 00_sensor_data_ingestion.ipynb first."
    )

# Use the first sensor file as the reference
reference_file = sensor_files[0]

# Read only the column names from the reference file
reference_columns = list(
    pd.read_csv(reference_file, nrows=0).columns
)

# Display the reference file and its columns
print(f"Reference file: {reference_file.name}")
print(reference_columns)

# Compare the column names of all sensor files
for file_path in sensor_files:

    # Read only the column names
    current_columns = list(
        pd.read_csv(file_path, nrows=0).columns
    )

    # Check whether the column names and order match
    if current_columns == reference_columns:
        print(f"{file_path.name}: Columns match")

    else:
        print(f"\n{file_path.name}: Columns do not match")

        # Find columns that are different
        extra_columns = set(current_columns) - set(reference_columns)
        missing_columns = set(reference_columns) - set(current_columns)

        print(f"Extra columns: {extra_columns}")
        print(f"Missing columns: {missing_columns}")

Reference file: On-street_Car_Parking_Sensor_Data_-_2011.csv
['DeviceId', 'ArrivalTime', 'DepartureTime', 'DurationSeconds', 'StreetMarker', 'Sign', 'Area', 'StreetId', 'StreetName', 'BetweenStreet1', 'BetweenStreet2', 'Side Of Street', 'In Violation', 'Vehicle Present']
On-street_Car_Parking_Sensor_Data_-_2011.csv: Columns match
On-street_Car_Parking_Sensor_Data_-_2012.csv: Columns match
On-street_Car_Parking_Sensor_Data_-_2013.csv: Columns match
On-street_Car_Parking_Sensor_Data_-_2014.csv: Columns match
On-street_Car_Parking_Sensor_Data_-_2015.csv: Columns match
On-street_Car_Parking_Sensor_Data_-_2016.csv: Columns match
On-street_Car_Parking_Sensor_Data_-_2017.csv: Columns match

On-street_Car_Parking_Sensor_Data_-_2018.csv: Columns do not match
Extra columns: {'SideOfStreet', 'InViolation', 'BetweenStreet2ID', 'SideName', 'AreaName', 'VehiclePresent', 'SignPlateID', 'BetweenStreet1ID', 'BayId', 'DurationMinutes'}
Missing columns: {'In Violation', 'Area', 'DurationSeconds', 'Vehicl

### 3. Schema Standardisation (Column Names + Duration + source_year)
- Rename the columns to create a consistent schema across all yearly datasets. 
- Convert parking duration values to seconds to ensure a consistent unit across all years.
- Add a `source_year` column to each yearly dataset to identify the original year of each record after all datasets are merged.

In [6]:
# Rename column in the 2011–2017
old_schema_rename = {
    "DeviceId": "device_id",
    "ArrivalTime": "arrival_time",
    "DepartureTime": "departure_time",
    "DurationSeconds": "duration_seconds",
    "StreetMarker": "street_marker",
    "Sign": "sign",
    "Area": "area",
    "StreetId": "street_id",
    "StreetName": "street_name",
    "BetweenStreet1": "between_street_1",
    "BetweenStreet2": "between_street_2",
    "Side Of Street": "side_of_street",
    "In Violation": "in_violation",
    "Vehicle Present": "vehicle_present"
}

# Rename column in the 2018–2020
new_schema_rename = {
    "DeviceId": "device_id",
    "ArrivalTime": "arrival_time",
    "DepartureTime": "departure_time",
    "DurationMinutes": "duration_minutes",
    "StreetMarker": "street_marker",
    "Sign": "sign",
    "AreaName": "area",
    "StreetId": "street_id",
    "StreetName": "street_name",
    "BetweenStreet1": "between_street_1",
    "BetweenStreet2": "between_street_2",
    "BetweenStreet1ID": "between_street_1_id",
    "BetweenStreet2ID": "between_street_2_id",
    "SideOfStreet": "side_of_street",
    "SideName": "side_name",
    "SideOfStreetCode": "side_of_street_code",
    "InViolation": "in_violation",
    "VehiclePresent": "vehicle_present",
    "BayId": "bay_id",
    "SignPlateID": "sign_plate_id"
}

In [ ]:
# Create a folder for processed datasets
processed_folder = project_path / "data" / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

# Standardise and save each yearly sensor dataset
for file_path in sensor_files:

    # Get the year from the file name
    year_match = re.search(r"20\d{2}", file_path.name)

    if year_match is None:
        print(f"Year not found: {file_path.name}")
        continue

    year = int(year_match.group())

    # Read the full yearly dataset
    df = pd.read_csv(file_path)

    # Standardise the column names
    if year <= 2017:
        df = df.rename(columns=old_schema_rename)

    else:
        df = df.rename(columns=new_schema_rename)

        # Convert duration from minutes to seconds
        df["duration_seconds"] = (
            pd.to_numeric(
                df["duration_minutes"],
                errors="coerce"
            ) * 60
        )

        # Remove the original duration column
        df = df.drop(columns=["duration_minutes"])

    # Add the source year
    df["source_year"] = year

    # Create the output file path
    output_file = processed_folder / f"parking_sensor_{year}_standardised.csv"

    # Save the standardised yearly dataset
    df.to_csv(output_file, index=False)

    # Display processing results
    print(
        f"{year}: Completed | "
        f"Columns: {len(df.columns)}"
    )

2011: Completed | Columns: 15
2012: Completed | Columns: 15
2013: Completed | Columns: 15
2014: Completed | Columns: 15
2015: Completed | Columns: 15
2016: Completed | Columns: 15
2017: Completed | Columns: 15


/var/folders/_w/v5cxbjrd6v12068_dmz5ysv80000gn/T/ipykernel_17181/130119316.py:20: DtypeWarning: Columns (3,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


2018: Completed | Columns: 20


/var/folders/_w/v5cxbjrd6v12068_dmz5ysv80000gn/T/ipykernel_17181/130119316.py:20: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


2019: Completed | Columns: 21


/var/folders/_w/v5cxbjrd6v12068_dmz5ysv80000gn/T/ipykernel_17181/130119316.py:20: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


2020: Completed | Columns: 20


In [ ]:
# Find all standardised sensor files
processed_files = sorted(
    processed_folder.glob("parking_sensor_*_standardised.csv")
)

# Display the number and names of processed files
print(f"Number of processed files: {len(processed_files)}")

for file_path in processed_files:
    print(file_path.name)

Number of processed files: 10
parking_sensor_2011_standardised.csv
parking_sensor_2012_standardised.csv
parking_sensor_2013_standardised.csv
parking_sensor_2014_standardised.csv
parking_sensor_2015_standardised.csv
parking_sensor_2016_standardised.csv
parking_sensor_2017_standardised.csv
parking_sensor_2018_standardised.csv
parking_sensor_2019_standardised.csv
parking_sensor_2020_standardised.csv


### 4. Validate Standardised Schema
Check that all yearly datasets use the same column names and data structure.

In [36]:
# Use the first processed file as the reference
reference_columns = list(
    pd.read_csv(processed_files[0], nrows=0).columns
)

# Compare processed column names across all years
for file_path in processed_files:

    current_columns = list(
        pd.read_csv(file_path, nrows=0).columns
    )

    if current_columns == reference_columns:
        print(f"{file_path.name}: Columns match")
    else:
        print(f"{file_path.name}: Columns do not match")

        extra_columns = set(current_columns) - set(reference_columns)
        missing_columns = set(reference_columns) - set(current_columns)

        print(f"Extra columns: {extra_columns}")
        print(f"Missing columns: {missing_columns}")

parking_sensor_2011_standardised.csv: Columns match
parking_sensor_2012_standardised.csv: Columns match
parking_sensor_2013_standardised.csv: Columns match
parking_sensor_2014_standardised.csv: Columns match
parking_sensor_2015_standardised.csv: Columns match
parking_sensor_2016_standardised.csv: Columns match
parking_sensor_2017_standardised.csv: Columns match
parking_sensor_2018_standardised.csv: Columns do not match
Extra columns: {'between_street_2_id', 'bay_id', 'side_name', 'between_street_1_id', 'sign_plate_id'}
Missing columns: set()
parking_sensor_2019_standardised.csv: Columns do not match
Extra columns: {'between_street_2_id', 'bay_id', 'side_name', 'between_street_1_id', 'sign_plate_id', 'side_of_street_code'}
Missing columns: set()
parking_sensor_2020_standardised.csv: Columns do not match
Extra columns: {'between_street_2_id', 'bay_id', 'side_name', 'between_street_1_id', 'sign_plate_id'}
Missing columns: set()


### 5. Merge Standardised Sensor Datasets (2011–2020)
Combine the standardised yearly datasets into a single dataset for further processing.

In [21]:
# Create the output file path
sensor = processed_folder / "parking_sensor_2011_2020.csv"

# Remove the existing output file before creating a new one
if sensor.exists():
    sensor.unlink()

# Create a list to store all column names 
all_columns = []

# Collect all column names from the standardised datasets
for file_path in processed_files:

    # Read only the column names
    current_columns = pd.read_csv(
        file_path,
        nrows=0
    ).columns.tolist()

    # Add new column names without creating duplicates
    for column in current_columns:
        if column not in all_columns:
            all_columns.append(column)

# Display the final set of columns
print(f"Total columns in the merged dataset: {len(all_columns)}")
print(all_columns)

# Write the column names only once
header_written = False

# Merge all standardised sensor datasets
for file_path in processed_files:

    print(f"Processing: {file_path.name}")

    # Read each dataset in chunks of 500,000 rows
    for chunk in pd.read_csv(
        file_path,
        chunksize=500_000,
        low_memory=False
    ):
        
        # Make all datasets use the same columns and column order
        # Missing columns are automatically filled with NaN
        chunk = chunk.reindex(columns=all_columns)

        # Add each chunk to the combined dataset
        chunk.to_csv(
            sensor,
            mode="a",
            header=not header_written,
            index=False
        )

        # Write the column names only once
        header_written = True

# Display the output file path
print("Master dataset saved successfully.")
print(sensor)

Total columns in the merged dataset: 21
['device_id', 'arrival_time', 'departure_time', 'duration_seconds', 'street_marker', 'sign', 'area', 'street_id', 'street_name', 'between_street_1', 'between_street_2', 'side_of_street', 'in_violation', 'vehicle_present', 'source_year', 'sign_plate_id', 'between_street_1_id', 'between_street_2_id', 'side_name', 'bay_id', 'side_of_street_code']
Processing: parking_sensor_2011_standardised.csv
Processing: parking_sensor_2012_standardised.csv
Processing: parking_sensor_2013_standardised.csv
Processing: parking_sensor_2014_standardised.csv
Processing: parking_sensor_2015_standardised.csv
Processing: parking_sensor_2016_standardised.csv
Processing: parking_sensor_2017_standardised.csv
Processing: parking_sensor_2018_standardised.csv
Processing: parking_sensor_2019_standardised.csv
Processing: parking_sensor_2020_standardised.csv
Master dataset saved successfully.
/Users/thanya/Desktop/Urban-Streetscape/data/processed/parking_sensor_2011_2020.csv


### 6. Load street_spatial + Check Street Name Format
Load the project street dataset and review the street name format to ensure it is consistent with the parking sensor dataset before matching and filtering.

In [23]:
# Read the project street dataset
street_spatial = pd.read_csv(
    data_folder / "street_spatial.csv"
)

# Check the first few rows
street_spatial.head(20)

,fid,street_name,street_segment_id,suburb,postcode,lga,treatment_or_control,intervention_type,cbd,metro,regional
0,1,KEILOR ROAD,211,Essendon,3040,Moonee Valley (C),control,control,0,1,0
1,2,KEILOR ROAD,212,Niddrie,3042,Moonee Valley (C),control,control,0,1,0
2,3,VICTORIA AVENUE,224,Albert Park,3206,Port Phillip (C),control,control,0,1,0
3,4,MALING ROAD,269,Canterbury,3126,Boroondara (C),control,control,0,1,0
4,5,STEPHENSONS ROAD,300,Mount Waverley,3149,Monash (C),control,control,0,1,0
5,6,PAKINGTON STREET,1041,Geelong West,3218,Greater Geelong (C),control,control,0,0,1
6,7,PAKINGTON STREET,1042,Geelong West,3218,Greater Geelong (C),control,control,0,0,1
7,8,PAKINGTON STREET,1043,Geelong West,3218,Greater Geelong (C),control,control,0,0,1
8,9,NEERIM ROAD,1398,Murrumbeena,3163,Glen Eira (C),control,control,0,1,0
9,10,POATH ROAD,1607,Hughesdale,3166,Monash (C),control,control,0,1,0


In [24]:
# Path to the merged dataset
sensor_file = processed_folder / "parking_sensor_2011_2020.csv"

# Read the first few rows
sensor = pd.read_csv(
    sensor_file,
    low_memory=False,
    nrows=20
)

sensor.head(20)

,device_id,arrival_time,departure_time,duration_seconds,street_marker,sign,area,street_id,street_name,between_street_1,...,side_of_street,in_violation,vehicle_present,source_year,sign_plate_id,between_street_1_id,between_street_2_id,side_name,bay_id,side_of_street_code
0,969,08/19/2011 06:12:58 PM,08/19/2011 06:13:03 PM,5,1404E,CW TOW M-F 16:00-19:00,Rialto,839,KING STREET,COLLINS STREET,...,2.0,False,False,2011,NaN,NaN,NaN,NaN,NaN,NaN
1,"1,183",09/13/2011 08:46:37 AM,09/13/2011 08:46:48 AM,11,1984N,1/2P M-SAT 7:30-19:30,Banks,528,COLLINS STREET,QUEEN STREET,...,3.0,False,False,2011,NaN,NaN,NaN,NaN,NaN,NaN
2,"1,457",09/09/2011 06:16:12 PM,09/09/2011 06:16:17 PM,5,1761S,LZ 30M M-F 7:30-19:30,Hyatt,669,FLINDERS LANE,RUSSELL STREET,...,4.0,False,False,2011,NaN,NaN,NaN,NaN,NaN,NaN
3,704,09/28/2011 11:55:29 AM,09/28/2011 11:56:12 AM,43,1194E,1/2P MTR M-F 7:30-16:30,Hardware,"1,171",QUEEN STREET,LONSDALE STREET,...,2.0,False,False,2011,NaN,NaN,NaN,NaN,NaN,NaN
4,"1,464",09/16/2011 05:22:39 PM,09/16/2011 05:31:33 PM,534,1843S,S/ No Stop M-F 16:00-19:30,Tavistock,669,FLINDERS LANE,WILLIAM STREET,...,4.0,False,False,2011,NaN,NaN,NaN,NaN,NaN,NaN
5,"2,055",10/14/2011 12:54:06 PM,10/14/2011 02:01:11 PM,"4,025",C3096,1P MTR M-SAT 7:30-19:30,Supreme,894,LONSDALE STREET,WILLIAM STREET,...,1.0,True,True,2011,NaN,NaN,NaN,NaN,NaN,NaN
6,"3,079",09/20/2011 06:00:11 AM,09/20/2011 10:57:43 AM,"17,852",4949W,1P MTR M-SAT 6:00-19:30,Victoria Market,"1,171",QUEEN STREET,VICTORIA STREET,...,5.0,True,True,2011,NaN,NaN,NaN,NaN,NaN,NaN
7,"1,271",11/09/2011 03:58:18 PM,11/09/2011 04:04:35 PM,377,2601S,1P TKT A M-SAT 7:30-19:30,County,907,Lt BOURKE STREET,KING STREET,...,4.0,False,False,2011,NaN,NaN,NaN,NaN,NaN,NaN
8,"3,629",11/02/2011 09:52:35 AM,11/02/2011 09:52:41 AM,6,7144N,1P RPA M-F 7:30-18:30,Queensberry,"1,391",WALSH STREET,KING STREET,...,3.0,False,False,2011,NaN,NaN,NaN,NaN,NaN,NaN
9,"1,812",10/31/2011 03:27:36 PM,10/31/2011 03:28:08 PM,32,3094N,2P DIS AOT 00:00-16:00,Supreme,894,LONSDALE STREET,WILLIAM STREET,...,3.0,False,False,2011,NaN,NaN,NaN,NaN,NaN,NaN


### 7. Filter Project Streets
Filter the merged dataset to include only the streets used in the project.

In [ ]:
# Standardise the project street names
street_spatial["street_name"] = (
    street_spatial["street_name"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create a set of unique project street names
project_streets = set(
    street_spatial["street_name"]
    .dropna()
    .unique()
)

# Path to the merged sensor dataset
sensor_file = processed_folder / "parking_sensor_2011_2020.csv"

# Path to the filtered sensor dataset
filtered_file = processed_folder / "parking_sensor_project_streets.csv"

# Remove the existing output file before creating a new one
if filtered_file.exists():
    filtered_file.unlink()

# Write the column names only once
header_written = False

# Read the merged sensor dataset in chunks
for chunk in pd.read_csv(
    sensor_file,
    chunksize=500_000,
    low_memory=False
):

    # Keep only streets included in the project
    filtered_chunk = chunk[
        chunk["street_name"].isin(project_streets)
    ]

    # Save the filtered records
    if not filtered_chunk.empty:

        filtered_chunk.to_csv(
            filtered_file,
            mode="a",
            header=not header_written,
            index=False
        )

        header_written = True

print("Filtered project street dataset saved successfully.")
print(filtered_file)


Filtered project street dataset saved successfully.
/Users/thanya/Desktop/Urban-Streetscape/data/processed/parking_sensor_project_streets.csv


### 8. Data Quality Check
Perform basic data quality checks to verify the merged dataset before data cleaning and exploratory data analysis.

##### Check Shape

In [29]:
# Count rows and get the number of columns
rows = 0
columns = None

for chunk in pd.read_csv(
    filtered_file,
    chunksize=500_000,
    low_memory=False
):

    rows += len(chunk)

    if columns is None:
        columns = len(chunk.columns)

print(f"Rows: {rows:,}")
print(f"Columns: {columns}")

Rows: 180,825,983
Columns: 21


In [ ]:
# Store all unique street names
unique_streets = set()

# Read the filtered dataset in chunks
for chunk in pd.read_csv(
    filtered_file,
    usecols=["street_name"],
    chunksize=500_000,
    low_memory=False
):

    unique_streets.update(
        chunk["street_name"]
        .dropna()
        .unique()
    )

# Display the number of unique street names
print(f"Number of unique street names in the filtered sensor dataset: {len(unique_streets)}")

Number of unique street names: 22
